# Spark NLP Healthcare Demo

Before beginning this Notebook, you **MUST** have the data volume mounted to an existing location.

Used the following guide to deploy Spark NLP (both JSL & OCR) to a Docker container. [GitHub - Docker Image for running Spark NLP for Healthcare Image](https://github.com/JohnSnowLabs/spark-nlp-workshop/blob/master/jupyter/docker_enterprise/docker_image_nlp_hc/sparknlp_for_healthcare_image/README.md)

Used the following guide from JohnSnowLabs to de-identify PDFs. [GitHub - 5.2 Spark OCR Deidentification](https://github.com/JohnSnowLabs/spark-nlp-workshop/blob/master/tutorials/Certification_Trainings/Healthcare/5.2.Spark_OCR_Deidentification.ipynb)


In [ ]:
import sparkocr
import pyspark
import sys
import sparknlp_jsl
from pyspark.sql import SparkSession
from sparkocr import start
import os
import base64
from sparkocr.transformers import *
from pyspark.ml import PipelineModel
from pyspark.sql import functions as F
from sparkocr.enums import *
from sparkocr.utils import display_images

from sparknlp.annotator import *
from sparknlp_jsl.annotator import *
from sparknlp.base import *

Apache Spark initializes and installs its dependencies.

In [ ]:
spark = sparkocr.start(
    secret=os.environ.get('SPARK_OCR_SECRET', ''),
    nlp_version=os.environ.get('PUBLIC_VERSION', ''),
    nlp_secret=os.environ.get('SECRET', ''),
    nlp_internal=os.environ.get('JSL_VERSION', '')
)
spark

Create a custom de-identification NLP pipeline. 

In [ ]:
def deidentification_nlp_pipeline(input_column, prefix = ""):
    document_assembler = DocumentAssembler() \
        .setInputCol(input_column) \
        .setOutputCol(prefix + "document")

    # Sentence Detector annotator, processes various sentences per line
    sentence_detector = SentenceDetector() \
        .setInputCols([prefix + "document"]) \
        .setOutputCol(prefix + "sentence")

    tokenizer = Tokenizer() \
        .setInputCols([prefix + "sentence"]) \
        .setOutputCol(prefix + "token")

    # Clinical word embeddings
    word_embeddings = WordEmbeddingsModel.pretrained("embeddings_clinical", "en", "clinical/models") \
        .setInputCols([prefix + "sentence", prefix + "token"]) \
        .setOutputCol(prefix + "embeddings")
    # NER model trained on i2b2 (sampled from MIMIC) dataset
    clinical_ner = MedicalNerModel.pretrained("ner_deid_large", "en", "clinical/models") \
        .setInputCols([prefix + "sentence", prefix + "token", prefix + "embeddings"]) \
        .setOutputCol(prefix + "ner")

    custom_ner_converter = NerConverterInternal() \
        .setInputCols([prefix + "sentence", prefix + "token", prefix + "ner"]) \
        .setOutputCol(prefix + "ner_chunk") \
        .setWhiteList(['NAME', 'AGE', 'CONTACT', 'LOCATION', 'PROFESSION', 'PERSON', 'DATE'])

    nlp_pipeline = Pipeline(stages=[
            document_assembler,
            sentence_detector,
            tokenizer,
            word_embeddings,
            clinical_ner,
            custom_ner_converter
        ])
    empty_data = spark.createDataFrame([[""]]).toDF(input_column)
    nlp_model = nlp_pipeline.fit(empty_data)
    return nlp_model

Create NLP Transformers for de-identifying DICOM images.

In [ ]:
# Read dicom as image
dicom_to_image = DicomToImage() \
    .setInputCol("content") \
    .setOutputCol("image_raw") \
    .setMetadataCol("metadata") \
    .setDeIdentifyMetadata(True)

adaptive_thresholding = ImageAdaptiveThresholding() \
    .setInputCol("image_raw") \
    .setOutputCol("corrected_image") \
    .setBlockSize(47) \
    .setOffset(4) \
    .setKeepInput(True)

# Extract text from image
ocr = ImageToText() \
    .setInputCol("corrected_image") \
    .setOutputCol("text")

# Found coordinates of sensitive data
position_finder = PositionFinder() \
    .setInputCols("ner_chunk") \
    .setOutputCol("coordinates") \
    .setPageMatrixCol("positions")

# Found sensitive data using DeIdentificationModel
deidentification_rules = DeIdentificationModel.pretrained("deidentify_rb_no_regex", "en", "clinical/models") \
    .setInputCols(["metadata_sentence", "metadata_token", "metadata_ner_chunk"]) \
    .setOutputCol("deidentified_metadata_raw")

finisher = Finisher() \
    .setInputCols(["deidentified_metadata_raw"]) \
    .setOutputCols("deidentified_metadata") \
    .setOutputAsArray(False) \
    .setValueSplitSymbol("") \
    .setAnnotationSplitSymbol("")

# Draw filled rectangle for hide sensitive data
drawRegions = ImageDrawRegions()  \
    .setInputCol("image_raw")  \
    .setInputRegionsCol("coordinates")  \
    .setOutputCol("image_with_regions")  \
    .setFilledRect(True) \
    .setRectColor(Color.black)

# Store image back to Dicom document
imageToDicom = ImageToDicom() \
    .setInputCol("image_with_regions") \
    .setOutputCol("dicom")

# OCR pipeline
deid_pipeline = PipelineModel(stages=[
    dicom_to_image,
    adaptive_thresholding,
    ocr,
    deidentification_nlp_pipeline(input_column="text"),
    position_finder,
    drawRegions,
    #imageToDicom  # Commented for able to demonstrate intermidiate results before aggregation
])

In [ ]:
from pathlib import Path

from pyspark.ml import Pipeline
from pyspark.sql import SparkSession
from pyspark.sql.functions import input_file_name

from sparkocr.transformers import *
from sparkocr.enums import *
from sparkocr.utils import display_image, display_images
from sparkocr.metrics import score
from sparknlp.annotator import *
from sparknlp_jsl.annotator import *
from sparknlp.base import *
import sparknlp_jsl

from sparkocr.transformers import *
from sparkocr.utils import display_image, to_pil_image

Create NLP Transformers for de-identifying PDF files.

In [ ]:
# Read Pdf as image
pdf_to_image = PdfToImage()\
    .setInputCol("content")\
    .setOutputCol("image_raw")\
    .setResolution(400)

image_to_pdf = ImageToPdf() \
    .setInputCol("image_with_regions") \
    .setOutputCol("pdf_bytes")

# Extract text from image
ocr = ImageToText() \
    .setInputCol("image_raw") \
    .setOutputCol("text") \
    .setIgnoreResolution(False) \
    .setPageIteratorLevel(PageIteratorLevel.SYMBOL) \
    .setPageSegMode(PageSegmentationMode.SPARSE_TEXT) \
    .setWithSpaces(True) \
    .setConfidenceThreshold(70)

hocr = ImageToHocr() \
    .setInputCol("image_raw") \
    .setOutputCol("hocr") \
    .setIgnoreResolution(False) \
    .setOcrParams(["preserve_interword_spaces=0"])\
    .setPageIteratorLevel(PageIteratorLevel.SYMBOL)\
    .setPageSegMode(PageSegmentationMode.SPARSE_TEXT) \

# Found coordinates of sensitive data
position_finder = PositionFinder() \
    .setInputCols("ner_chunk") \
    .setOutputCol("coordinates") \
    .setPageMatrixCol("positions")

# Draw filled rectangle for hide sensitive data
drawRegions = ImageDrawRegions()  \
    .setInputCol("image_raw")  \
    .setInputRegionsCol("coordinates")  \
    .setOutputCol("image_with_regions")  \
    .setFilledRect(True) \
    .setRectColor(Color.black)

# OCR pipeline
deid_pipeline = PipelineModel(stages=[
    pdf_to_image,
    ocr,
    hocr,
    deidentification_nlp_pipeline(input_column="text"),
    position_finder,
    drawRegions
])

Create folder structure for input and output PDFs.

Read the PDF files into Spark as binary files. Coalesce to only 4 partitions to avoid HEAP overflow.

Co

In [ ]:
input_folder  = "/data/input"
output_folder = "/data/output"
os.makedirs(output_folder, exist_ok=True)

pdfs = (
    spark.read
         .format("binaryFile")
         .load(f"{input_folder}/*.pdf")
         .withColumn("orig_path", input_file_name())
)
pdfs = pdfs.coalesce(4) # only 4 partitions at a time

processed = deid_pipeline.transform(pdfs)
result = (
    image_to_pdf
      .transform(processed)
      .withColumnRenamed("path", "orig_path")
      .select("orig_path", "pdf_bytes")
)

for row in result.collect():
    src = Path(row.orig_path)
    out = Path(output_folder) / src.name
    with open(out, "wb") as f:
        f.write(row.pdf_bytes)

print(f"Wrote de-ID’d PDFs to {output_folder}")